Author: Kaili Fan

This notebook demonstrates how to run the ATAC_QC pipeline from raw fragments through cell QC, doublet removal, clustering, merging, and UMAP in a reproducible way.

In [1]:
import sys
import os

repo_root = os.path.abspath(os.path.join(".."))  # one level up from notebooks/
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [2]:
os.getcwd()

'/data/fankaili/scripts/multi-Omic_workflow/ATAC_QC/notebooks'

In [3]:
os.chdir('../')
os.getcwd()

'/data/fankaili/scripts/multi-Omic_workflow/ATAC_QC'

## Config overview

We store all parameters, file paths in `config/atac_qc_config.yaml`:

In [4]:
from atac_qc.utils import load_config

config_path = "config/atac_qc_config.yaml"
config = load_config(config_path)
import yaml
print(yaml.dump(config, default_flow_style=False))

params:
  min_fragments: 100
  n_threads: 32
  tissue: Kidney
paths:
  fragment_dir: /data/fankaili/project/dgtex/data/fragment_files/
  output_figures_dir: /data/fankaili/project/dgtex/QC/20250414/Kidney/atac_figures/
  output_h5ad_dir: /data/fankaili/project/dgtex/QC/20250414/Kidney/atac_h5ad/
  sample_metadata: /data/fankaili/project/dgtex/data/data_metatable_20250414.txt
  workdir: /data/fankaili/project/dgtex/QC/20250414/Kidney/
process:
  n_features: 250000
qc:
  atac_qc_cutoff_table: /data/fankaili/project/dgtex/QC/dGTEx_snATAC_QC_cutoffs.xlsx
  sheet_name: 20250414
references:
  barcode_whitelist: /data/fankaili/ref/10x_barcode/10XMultiOmic_737K_arc-v1.txt
  gencode_gtf: /data/fankaili/ref/GENCODE/gencode.v47.primary_assembly.annotation.gtf



If you are running on new data, ensure data are formatted like those in `test_data/` and config paths are updated.

## step1 - Remove PCR Chimeric Reads

Below will automatically run scripts/run_rmPCRchimeric_parallel.py for all samples

In [ ]:
!python scripts/run_rmPCRchimeric_parallel.py --fragdir test_data/fragment_files \
  --script scripts/remove_pcr_chimeric.py --metadata test_data/test_metadata.txt --threads 2

## step2 - Convert Fragment Files to h5ad

In [ ]:
!python scripts/load_fragments.py config/atac_qc_config.yaml

## step3 - QC Filtering and Visualization

In [ ]:
!python scripts/qc_kde_filter.py config/atac_qc_config.yaml demo_run

## step4 - Doublet Detection and Plotting

In [ ]:
!python scripts/processing_doublet_detection.py config/atac_qc_config.yaml demo_run

Potentially, you might need to check some doublet cutoffs and decide whether use default (midpoint of the double probabilities).

## step5 - Clustering and Downstream Analysis

In [ ]:
!python scripts/downstream_processing.py config/atac_qc_config.yaml demo_run

## step6 - Merge Samples and Joint UMAP

In [5]:
!python scripts/concatenate_and_cluster.py config/atac_qc_config.yaml demo_run

Traceback (most recent call last):
  File "/home/fankaili/miniconda3/envs/workenv/lib/python3.12/site-packages/llvmlite/binding/ffi.py", line 136, in __getattr__
    return self._fntab[name]
           ~~~~~~~~~~~^^^^^^
KeyError: 'LLVMPY_AddSymbol'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/fankaili/miniconda3/envs/workenv/lib/python3.12/site-packages/llvmlite/binding/ffi.py", line 117, in _load_lib
    self._lib_handle = ctypes.CDLL(str(lib_path))
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/fankaili/miniconda3/envs/workenv/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /lib64/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /home/fankaili/miniconda3/envs/workenv/lib/python3.12/site-packages/llvmlite/binding/libllvmlite.so)

The above exception was the direct cause of 